# UE9 - Seance 6 - Variant Calling (Correction)

**Auteur : Marwa Zidi** — Universite Paris Cite

**Date : 11/02/2026**

---

**Contexte clinique** :

| Echantillon | Prelevement | Envahissement tumoral |
|-------------|-------------|----------------------|
| `Diagnosis` | Moelle osseuse au diagnostic | 30% de blastes |
| `Germline` | Biopsie cutanee — fibroblastes | Aucun |

## Etape 1 — Controle qualite (FastQC)

---

#### Affichez le contenu du dossier `Raw_data`

In [ ]:
ls Raw_data

#### Pourquoi deux fichiers FASTQ par echantillon ?

#### Premieres lignes d'un fichier FASTQ

In [ ]:
head -n 4 Raw_data/Diagnosis_R1.fastq

#### Description des 4 lignes d'un read FASTQ

In [ ]:
mkdir Quality_control

In [ ]:
# FastQC sur tous les fichiers FASTQ d'un coup avec le wildcard *
fastqc --outdir Quality_control Raw_data/*.fastq

## Etape 2 — Alignement (BWA)

---

### Genome de reference (deja indexe)

In [ ]:
mkdir Intermediate_files

In [ ]:
# Alignement Diagnosis
bwa mem \
Reference_genome/hg38.fa \
Raw_data/Diagnosis_R1.fastq \
Raw_data/Diagnosis_R2.fastq > Intermediate_files/Diagnosis.sam

In [ ]:
samtools view Intermediate_files/Diagnosis.sam | head -n 1

#### Description des 6 premiers champs SAM

In [ ]:
# Alignement Germline
bwa mem \
Reference_genome/hg38.fa \
Raw_data/Germline_R1.fastq \
Raw_data/Germline_R2.fastq > Intermediate_files/Germline.sam

## Etape 3 — Samtools : compression, tri, indexation

---

In [ ]:
samtools view -b Intermediate_files/Diagnosis.sam > Intermediate_files/Diagnosis.bam

In [ ]:
samtools view -b Intermediate_files/Germline.sam > Intermediate_files/Germline.bam

In [ ]:
samtools sort Intermediate_files/Diagnosis.bam -o Intermediate_files/Diagnosis_sorted.bam

In [ ]:
samtools sort Intermediate_files/Germline.bam -o Intermediate_files/Germline_sorted.bam

In [ ]:
ls -lh Intermediate_files/

#### Differences de taille SAM / BAM / sorted.BAM

In [ ]:
samtools --help

In [ ]:
samtools index Intermediate_files/Diagnosis_sorted.bam
samtools index Intermediate_files/Germline_sorted.bam

## Etape 4 — Qualite d'alignement (Qualimap)

---

In [ ]:
export JAVA_OPTS="-Xmx1200M"
qualimap bamqc \
-bam Intermediate_files/Diagnosis_sorted.bam \
-outdir Quality_control/Diagnosis_bamqc \
-gff Reference_genome/Genes_exons.gtf

In [ ]:
samtools view -c -q 60 Intermediate_files/Diagnosis_sorted.bam

In [ ]:
qualimap bamqc \
-bam Intermediate_files/Germline_sorted.bam \
-outdir Quality_control/Germline_bamqc \
-gff Reference_genome/Genes_exons.gtf

samtools view -c -q 60 Intermediate_files/Germline_sorted.bam

#### Observation IGV — gene IDH1

## Etape 5 — Appel de variants (VarScan)

---

In [ ]:
samtools mpileup -f Reference_genome/hg38.fa \
Intermediate_files/Diagnosis_sorted.bam > Intermediate_files/Diagnosis.mpileup

In [ ]:
head -n 175 Intermediate_files/Diagnosis.mpileup | tail -n 10

#### Description des 6 colonnes du mpileup

In [ ]:
samtools mpileup -f Reference_genome/hg38.fa \
Intermediate_files/Germline_sorted.bam > Intermediate_files/Germline.mpileup

In [ ]:
mkdir Results

In [ ]:
varscan mpileup2cns --help

#### Appel permissif (exploration)

In [ ]:
varscan mpileup2cns \
Intermediate_files/Diagnosis.mpileup \
--min-coverage 1 \
--min-reads2 1 \
--min-var-freq 0.001 \
--min-avg-qual 1 \
--p-value 1 \
--strand-filter 0 \
--output-vcf 1 \
--variants \
> Results/Diagnosis.vcf

#### Appel avec criteres stringents (recommandes pour panel cible)

In [ ]:
varscan mpileup2cns \
Intermediate_files/Diagnosis.mpileup \
--min-coverage 10 \
--min-reads2 2 \
--min-var-freq 0.02 \
--min-avg-qual 20 \
--p-value 0.01 \
--strand-filter 1 \
--output-vcf 1 \
--variants \
> Results/Diagnosis.vcf

In [ ]:
head -n 33 Results/Diagnosis.vcf | tail

In [ ]:
# VarScan sur Germline avec les memes criteres
varscan mpileup2cns \
Intermediate_files/Germline.mpileup \
--min-coverage 10 \
--min-reads2 2 \
--min-var-freq 0.02 \
--min-avg-qual 20 \
--p-value 0.01 \
--strand-filter 1 \
--output-vcf 1 \
--variants \
> Results/Germline.vcf

#### Types de variants non detectes par VarScan

## Etape 6 — Annotation (VEP)

---

In [ ]:
/opt/ensembl-vep/vep --help

In [ ]:
/opt/ensembl-vep/vep -i Results/Diagnosis.vcf \
    -o Results/Diagnosis_vep.vcf \
    --vcf \
    --database \
    --assembly GRCh38 \
    --fasta Reference_genome/hg38.fa \
    --pick \
    --symbol \
    --hgvs \
    --af \
    --polyphen b \
    --sift b

In [ ]:
/opt/ensembl-vep/vep -i Results/Germline.vcf \
    -o Results/Germline_vep.vcf \
    --vcf \
    --database \
    --assembly GRCh38 \
    --fasta Reference_genome/hg38.fa \
    --pick \
    --symbol \
    --hgvs \
    --af \
    --polyphen b \
    --sift b

## Etape 7 — Filtrage (R)

---

In [ ]:
Rscript Scripts/VCF_to_TSV.R Results/Diagnosis_vep.vcf

In [ ]:
Rscript Scripts/Variants_filtering.R Results/Diagnosis_vep.vcf.tsv

#### Interpretation des variants retenus — Diagnosis

In [ ]:
Rscript Scripts/VCF_to_TSV.R Results/Germline_vep.vcf
Rscript Scripts/Variants_filtering.R Results/Germline_vep.vcf.tsv

#### Interpretation des variants retenus — Germline

## Conclusion

---

#### Recapitulatif du pipeline

### Pour aller plus loin

Les variants pathogenes de **DDX41** sont associes aux syndromes myelodysplasiques et aux LAM chez les patients de plus de 60 ans.
La decouverte d'un tel variant a des implications pour la prise en charge du patient ET pour le depistage de ses apparentes.

*Germline DDX41 mutations define a significant entity within adult MDS/AML patients.* Sebert et al. Blood 2019
https://pubmed.ncbi.nlm.nih.gov/31484648/